# R2AI ANDGATE Kaggle Pipeline

Notebook này dùng để chạy repo `danhyoyo/r2ai_andgate` trên Kaggle: clone repo, cài dependency, tạo corpus luật, build index, chạy test và xuất `results.json` / `submission.zip`.

Khuyến nghị trên Kaggle free: chạy `USE_LLM = False` trước để lấy kết quả chắc chắn bằng BM25 + BGE-M3 + reranker. Chỉ bật `USE_LLM = True` khi GPU đủ VRAM và còn nhiều thời gian runtime.

## 0. Kaggle Setup

Trong Kaggle Notebook, bật:

- `Accelerator`: GPU.
- `Internet`: On, nếu cần clone GitHub và tải Hugging Face model/dataset.

Nếu cuộc thi hoặc notebook tắt Internet, hãy upload repo/corpus/model thành Kaggle Dataset rồi copy từ `/kaggle/input/...`.

In [ ]:

from pathlib import Path
import glob
import json
import os
import shutil
import subprocess
import sys
import time


def run_cmd(cmd, *, required=False, label=""):
    """Run a command without turning optional Kaggle steps into red cells."""
    title = label or " ".join(str(part) for part in cmd[:4])
    print("Running:", " ".join(str(part) for part in cmd))
    try:
        result = subprocess.run([str(part) for part in cmd], check=False)
    except FileNotFoundError as exc:
        print(f"WARNING: {title} could not start: {exc}")
        if required:
            raise
        return False
    if result.returncode != 0:
        print(f"WARNING: {title} exited with code {result.returncode}")
        if required:
            raise RuntimeError(f"Required command failed: {title}")
        return False
    return True


def count_lines(path):
    try:
        with Path(path).open("r", encoding="utf-8") as handle:
            return sum(1 for _ in handle)
    except FileNotFoundError:
        return 0


IS_KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/danhyoyo/r2ai_andgate.git"
BRANCH = "main"
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd().resolve()
REPO_DIR = (WORK_DIR / "r2ai_andgate") if IS_KAGGLE else Path.cwd().resolve()

# Corpus import. Increase IMPORT_DOCS for a larger corpus; use -1 only when runtime is stable.
FORCE_IMPORT = False
IMPORT_DOCS = 5000
IMPORT_CHUNK_SIZE = 1000
METADATA_SCAN_LIMIT = 50000
IMPORT_BATCH_SIZE = 100
IMPORT_BACKEND = "parquet"

# Inference mode. Optional model features automatically fall back if they fail.
VARIANT = "balanced"          # recall | balanced | precision
USE_DENSE = True              # falls back to BM25 if unavailable
USE_CROSS_ENCODER = True      # falls back to rule reranker if unavailable
USE_LLM = False               # keep False on small/free GPUs unless you have enough VRAM
LOG_EVERY = 10
CHECKPOINT_EVERY = 10

# Runtime safeguards for 16GB Kaggle GPUs.
R2AI_RERANK_DEVICE = "auto"    # set "cpu" if cross-encoder still OOMs
R2AI_RERANK_BATCH_SIZE = "2"
R2AI_MIN_RERANK_FREE_GB = "6"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_DIR = WORK_DIR / "models" / "Qwen2.5-7B-Instruct"

# Sharding avoids long single runs. Example: NUM_SHARDS=4 and SHARD_ID=0,1,2,3.
NUM_SHARDS = 1
SHARD_ID = 0

ARTICLES_PATH = Path("data/processed/articles.jsonl")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
test_path = Path("data/test/test.json")
output_path = RESULTS_DIR / "results.json"

SKIP_SETUP = False
SKIP_INFERENCE = False
SKIP_VALIDATE = False

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["R2AI_RERANK_DEVICE"] = R2AI_RERANK_DEVICE
os.environ["R2AI_RERANK_BATCH_SIZE"] = R2AI_RERANK_BATCH_SIZE
os.environ["R2AI_MIN_RERANK_FREE_GB"] = R2AI_MIN_RERANK_FREE_GB

print("Config ready")
print("Kaggle:", IS_KAGGLE)
print("Work dir:", WORK_DIR)


## 1. Clone repo từ GitHub

In [ ]:

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    if IS_KAGGLE:
        run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], required=False, label="git pull")
else:
    ok = run_cmd(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], required=False, label="git clone")
    if not ok and not REPO_DIR.exists():
        SKIP_SETUP = True
        print("Repo is not available. Upload the repo as a Kaggle Dataset or enable Internet, then rerun from this cell.")

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    print("Repo:", Path.cwd())
    run_cmd(["git", "rev-parse", "--short", "HEAD"], required=False, label="git rev-parse")
else:
    SKIP_SETUP = True
    print("Skipping later cells because the repo directory is missing.")


## 2. Cài dependency và kiểm tra GPU

In [ ]:

if SKIP_SETUP:
    print("Skipped dependency setup because repo setup did not finish.")
else:
    packages = [
        "numpy",
        "transformers",
        "sentence-transformers",
        "huggingface-hub",
        "datasets",
        "pyarrow",
        "accelerate",
        "safetensors",
    ]
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], required=False, label="pip install optional packages")

    try:
        import torch
        print("torch:", torch.__version__)
        print("cuda:", torch.cuda.is_available())
        if torch.cuda.is_available():
            print("gpu:", torch.cuda.get_device_name(0))
            print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    except Exception as exc:
        print("Torch/GPU check warning:", exc)

    run_cmd([sys.executable, "-m", "src.models.check_environment"], required=False, label="environment check")


## 3. Chuẩn bị corpus luật

Notebook ưu tiên dùng `articles.jsonl` nếu bạn đã upload sẵn trong Kaggle Dataset. Nếu không có, nó sẽ import từ Hugging Face theo từng chunk để có thể resume tốt hơn.

In [ ]:

if SKIP_SETUP:
    SKIP_INFERENCE = True
    print("Skipped corpus preparation because repo setup did not finish.")
else:
    Path("data/processed").mkdir(parents=True, exist_ok=True)

    input_root = Path("/kaggle/input")
    uploaded_articles = sorted(input_root.glob("**/articles.jsonl")) if input_root.exists() else []
    if uploaded_articles and not FORCE_IMPORT:
        source = uploaded_articles[0]
        shutil.copy2(source, ARTICLES_PATH)
        print("Copied uploaded corpus:", source, "->", ARTICLES_PATH)
    elif ARTICLES_PATH.exists() and not FORCE_IMPORT:
        print("Using existing corpus:", ARTICLES_PATH.resolve())
    else:
        temp_path = ARTICLES_PATH.with_name(f"{ARTICLES_PATH.stem}.importing{ARTICLES_PATH.suffix}")
        if temp_path.exists():
            temp_path.unlink()

        import_ok = True
        if IMPORT_DOCS == -1:
            cmd = [
                sys.executable, "-m", "src.preprocess.import_hf_legal",
                "--backend", IMPORT_BACKEND,
                "--output", str(temp_path),
                "--max-docs", "-1",
                "--metadata-scan-limit", "-1",
                "--batch-size", str(IMPORT_BATCH_SIZE),
                "--require-metadata",
            ]
            import_ok = run_cmd(cmd, required=False, label="full corpus import")
        else:
            for offset in range(0, IMPORT_DOCS, IMPORT_CHUNK_SIZE):
                max_docs = min(IMPORT_CHUNK_SIZE, IMPORT_DOCS - offset)
                cmd = [
                    sys.executable, "-m", "src.preprocess.import_hf_legal",
                    "--backend", IMPORT_BACKEND,
                    "--output", str(temp_path),
                    "--offset", str(offset),
                    "--max-docs", str(max_docs),
                    "--metadata-scan-limit", str(METADATA_SCAN_LIMIT),
                    "--batch-size", str(IMPORT_BATCH_SIZE),
                    "--require-metadata",
                    "--append",
                ]
                print("Import chunk offset", offset, "size", max_docs)
                if not run_cmd(cmd, required=False, label=f"corpus import chunk {offset}"):
                    import_ok = False
                    break

        if temp_path.exists() and count_lines(temp_path) > 0:
            shutil.move(str(temp_path), str(ARTICLES_PATH))
            print("Imported corpus rows:", count_lines(ARTICLES_PATH))
        elif ARTICLES_PATH.exists():
            print("Import did not finish; keeping existing corpus:", ARTICLES_PATH.resolve())
        else:
            SKIP_INFERENCE = True
            print("No corpus is available. Upload articles.jsonl or enable Internet, then rerun this cell.")

    if ARTICLES_PATH.exists():
        print("Article rows:", count_lines(ARTICLES_PATH))
        print("Corpus path:", ARTICLES_PATH.resolve())
    else:
        SKIP_INFERENCE = True
        print("Corpus path is still missing; later inference cells will be skipped.")


## 4. Build BM25 index

In [ ]:

if SKIP_INFERENCE or not ARTICLES_PATH.exists():
    print("Skipped BM25 build because the corpus is missing.")
else:
    run_cmd([
        sys.executable, "-m", "src.retrieval.bm25_retriever", "build",
        "--articles", str(ARTICLES_PATH),
        "--output", "indexes/bm25/bm25.pkl",
    ], required=False, label="BM25 build")


## 5. Smoke test retrieval

In [ ]:

if SKIP_INFERENCE or not ARTICLES_PATH.exists():
    print("Skipped smoke test because the corpus is missing.")
else:
    run_cmd([
        sys.executable, "-m", "src.retrieval.pipeline",
        "--articles", str(ARTICLES_PATH),
        "--question", "Doanh nghiep nho va vua duoc ho tro khi dap ung dieu kien nao?",
        "--variant", VARIANT,
    ], required=False, label="retrieval smoke test")


## 6. Tạo test shard để tránh timeout

Nếu sợ hết 12 tiếng, đặt `NUM_SHARDS = 4` rồi chạy 4 session với `SHARD_ID = 0, 1, 2, 3`. Mỗi session tạo một file `results_shard_*`. Sau đó dùng cell merge ở cuối.

In [ ]:

if SKIP_SETUP:
    SKIP_INFERENCE = True
    print("Skipped test shard setup because repo setup did not finish.")
elif not Path("data/test/test.json").exists():
    SKIP_INFERENCE = True
    print("Missing data/test/test.json. Upload the test file, then rerun from this cell.")
else:
    test_records = json.loads(Path("data/test/test.json").read_text(encoding="utf-8"))
    NUM_SHARDS = max(1, int(NUM_SHARDS))
    if SHARD_ID < 0 or SHARD_ID >= NUM_SHARDS:
        print(f"SHARD_ID={SHARD_ID} is outside 0..{NUM_SHARDS - 1}; using SHARD_ID=0.")
        SHARD_ID = 0

    if NUM_SHARDS > 1:
        shard_records = [row for index, row in enumerate(test_records) if index % NUM_SHARDS == SHARD_ID]
        test_path = Path(f"data/test/test_shard_{SHARD_ID}_of_{NUM_SHARDS}.json")
        test_path.write_text(json.dumps(shard_records, ensure_ascii=False, indent=2), encoding="utf-8")
        output_path = RESULTS_DIR / f"results_shard_{SHARD_ID}_of_{NUM_SHARDS}.json"
    else:
        test_path = Path("data/test/test.json")
        output_path = RESULTS_DIR / "results.json"

    print("Test path:", test_path, "records:", len(json.loads(test_path.read_text(encoding="utf-8"))))
    print("Output path:", output_path)


## 7. Optional: tải Qwen local

Chỉ chạy cell này khi `USE_LLM = True`. Trên GPU 16GB, khả năng OOM cao; trên 24GB sẽ ổn hơn.

In [ ]:

model_args = []
if USE_LLM and not SKIP_INFERENCE:
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    ok = run_cmd([
        "huggingface-cli", "download", MODEL_ID,
        "--local-dir", str(MODEL_DIR),
    ], required=False, label="Qwen download")
    if ok:
        model_args = ["--model-path", str(MODEL_DIR)]
        print("Using local LLM:", MODEL_DIR)
    else:
        USE_LLM = False
        print("Qwen download failed; continuing with extractive fallback.")
else:
    print("USE_LLM=False: using extractive grounded answer fallback.")


## 8. Chạy inference tạo kết quả

In [ ]:

if SKIP_INFERENCE:
    SKIP_VALIDATE = True
    print("Skipped inference because an earlier required input is missing.")
else:
    cmd = [
        sys.executable, "-u", "-m", "src.submit.build_results",
        "--test", str(test_path),
        "--articles", str(ARTICLES_PATH),
        "--output", str(output_path),
        "--variant", VARIANT,
        "--log-every", str(LOG_EVERY),
        "--checkpoint-every", str(CHECKPOINT_EVERY),
    ]
    if USE_DENSE:
        cmd.append("--use-dense")
    if USE_CROSS_ENCODER:
        cmd.append("--use-cross-encoder")
    cmd.extend(model_args)

    start = time.time()
    ok = run_cmd(cmd, required=False, label="build results")
    partial_path = output_path.with_name(f"{output_path.stem}.partial{output_path.suffix}")
    if not output_path.exists() and partial_path.exists():
        shutil.copy2(partial_path, output_path)
        print("Copied partial output to final output:", partial_path, "->", output_path)
    if not output_path.exists():
        SKIP_VALIDATE = True
        print("No output file was produced. Check the warnings above before validating.")
    print("Inference cell finished in minutes:", round((time.time() - start) / 60, 2), "success:", ok)


## 9. Validate và zip

Nếu chạy shard thì mỗi shard chỉ validate file shard. Chỉ zip submission cuối khi đã merge đủ shard.

In [ ]:

if SKIP_VALIDATE or not output_path.exists():
    print("Skipped validate/zip because no output file is available yet.")
else:
    run_cmd([
        sys.executable, "-m", "src.submit.validate_results",
        "--input", str(output_path),
        "--output", str(output_path),
        "--repair",
    ], required=False, label="repair results")
    run_cmd([sys.executable, "-m", "src.submit.validate_results", "--input", str(output_path)], required=False, label="validate results")

    if NUM_SHARDS == 1:
        run_cmd([
            sys.executable, "-m", "src.submit.zip_submission",
            "--results", str(output_path),
            "--output", "results/submission.zip",
        ], required=False, label="zip submission")
        print("Final files:", output_path, Path("results/submission.zip"))
    else:
        print("Shard file is ready:", output_path)
        print("Run all shard IDs, then merge them with the next cell.")


## 10. Merge shard outputs nếu chia nhiều session

Copy tất cả `results_shard_*_of_*.json` vào thư mục `results/` rồi chạy cell này.

In [ ]:

shard_paths = sorted(Path("results").glob("results_shard_*_of_*.json"))
if shard_paths:
    merged = []
    seen_ids = set()
    for path in shard_paths:
        try:
            rows = json.loads(path.read_text(encoding="utf-8"))
        except Exception as exc:
            print("Skipping unreadable shard:", path, exc)
            continue
        for row in rows:
            row_id = row.get("id")
            if row_id not in seen_ids:
                seen_ids.add(row_id)
                merged.append(row)
    merged.sort(key=lambda item: str(item.get("id", "")))
    final_path = Path("results/results.json")
    final_path.write_text(json.dumps(merged, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Merged records:", len(merged), "->", final_path)
    run_cmd([sys.executable, "-m", "src.submit.validate_results", "--input", str(final_path), "--output", str(final_path), "--repair"], required=False, label="repair merged results")
    run_cmd([sys.executable, "-m", "src.submit.validate_results", "--input", str(final_path)], required=False, label="validate merged results")
    run_cmd([sys.executable, "-m", "src.submit.zip_submission", "--results", str(final_path), "--output", "results/submission.zip"], required=False, label="zip merged submission")
else:
    print("No shard files found.")


## 11. Xem nhanh output

In [ ]:

paths_to_show = [Path("results/results.json"), Path("results/submission.zip"), output_path, ARTICLES_PATH]
for path in paths_to_show:
    if path.exists():
        print(path, round(path.stat().st_size / 1024 / 1024, 2), "MB")

preview_path = Path("results/results.json") if Path("results/results.json").exists() else output_path
if preview_path.exists():
    preview = json.loads(preview_path.read_text(encoding="utf-8"))[:2]
    preview
else:
    print("No preview yet because no results file exists.")
